# The confound, and what removing it costs

`data/eval/` pairs COCO val2017 (real) against DALL-E Advanced (fake). The two
halves did not just come from different generators -- they came from different
*encoders*. This notebook fits a classifier that is forbidden from looking at a
single pixel: it sees only the JPEG quantisation tables, the chroma subsampling
mode, the dimensions, and the file size. Whatever accuracy it reaches is
accuracy that has nothing to do with detecting generated imagery.

Then it re-encodes both halves through one encoder at one quality and one crop
size, and asks the same question again. The gap between the two numbers is the
part of this benchmark that is an artifact of how the files were saved.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import collections
import numpy as np
from omegaconf import OmegaConf

from provenance.shortcut import (
    collect, compare, featurise, fit_probe, format_table, read_structure,
)

cfg = OmegaConf.load("configs/default.yaml")
SEED = int(cfg.seed)
np.random.seed(SEED)
print(f"seed {SEED}  |  sources {list(cfg.shortcut.sources)}  |  cap {cfg.shortcut.max_per_class}/class")

seed 1337  |  sources ['coco_val2017', 'dalle_advanced']  |  cap 2000/class


## 1. What the header alone reveals

Before fitting anything, look at the raw container facts, split by label. No
decoding: `read_structure` parses JPEG headers and stops.

In [2]:
rows = collect(cfg, max_per_class=int(cfg.shortcut.max_per_class))
root = str(cfg.paths.root)

SUBS = {0: "4:4:4", 1: "4:2:2", 2: "4:2:0", -1: "n/a"}
breakdown = collections.defaultdict(collections.Counter)
sizes = collections.defaultdict(collections.Counter)
for r in rows:
    s = read_structure(os.path.join(root, r["path"]))
    breakdown[r["label"]][(s["format"], SUBS[s["subsampling"]])] += 1
    sizes[r["label"]][f'{s["width"]}x{s["height"]}'] += 1

names = {0: "real (COCO)", 1: "fake (DALL-E)"}
for label in sorted(breakdown):
    total = sum(breakdown[label].values())
    print(f"{names[label]}  n={total}")
    for (fmt, sub), count in breakdown[label].most_common():
        print(f"    {fmt:<5} {sub:<6} {count:>5}  ({100 * count / total:5.1f}%)")
    print(f"    most common sizes: {', '.join(f'{k} x{v}' for k, v in sizes[label].most_common(3))}")
    print()

real (COCO)  n=2000
    jpeg  4:4:4   1995  ( 99.8%)
    jpeg  n/a        5  (  0.2%)
    most common sizes: 640x480 x447, 640x427 x243, 480x640 x126

fake (DALL-E)  n=2000
    jpeg  4:2:0   1859  ( 93.0%)
    png   n/a      139  (  7.0%)
    webp  n/a        2  (  0.1%)
    most common sizes: 1024x1024 x1704, 1792x1024 x234, 1024x1792 x22



The classes are disjoint on a field that no generator controls. COCO is
uniformly 4:4:4 chroma; DALL-E is uniformly 4:2:0, plus a slice of files that
carry a `.jpg` name but are actually PNG or WebP inside. Resolution separates
them a second time -- COCO is camera-shaped and small, DALL-E is square and
1024-plus. Either field on its own is close to a giveaway.

## 2. Fit the content-blind probe

Logistic regression on 138 features: 64 luma quantisation coefficients, 64
chroma, a one-hot subsampling mode, width, height, aspect ratio,
bytes-per-pixel, and two flags marking which of those were absent. Scores are
out-of-fold from stratified 5-fold CV, so this is not a fit reported on itself.

In [3]:
X, y = featurise(rows, root=root)
raw = fit_probe(X, y, seed=SEED, folds=int(cfg.shortcut.folds),
                C=float(cfg.shortcut.C), max_iter=int(cfg.shortcut.max_iter))

print(f"ROC AUC  {raw['auc']:.4f}   (per fold: {', '.join(f'{v:.4f}' for v in raw['auc_per_fold'])})")
print(f"n = {raw['n']}  ({raw['n_positive']} fake)   features = {raw['n_features']}\n")
print("strongest coefficients (standardised):")
for f in raw["top_features"][:8]:
    print(f"    {f['name']:<22}{f['coef']:>8.3f}")

ROC AUC  1.0000   (per fold: 1.0000, 1.0000, 1.0000, 1.0000, 1.0000)
n = 4000  (2000 fake)   features = 138

strongest coefficients (standardised):
    height                   1.362
    subsampling_444         -1.273
    width                    1.271
    subsampling_420          1.035
    is_jpeg                 -0.985
    has_chroma_table        -0.643
    subsampling_other        0.643
    bytes_per_pixel          0.394


### Interpretation

**AUC 1.000, on every fold.** A model that has never seen a pixel separates
real from fake perfectly, and it does so on fields that describe the encoder,
not the image: which chroma subsampling mode was chosen, what quantisation
table the encoder used, how many pixels the file has.

This is what any detector trained and evaluated on this pairing is being
rewarded for. It can score near-perfectly by learning "4:2:0 and square means
fake" and never learn anything about generated imagery at all. The failure is
silent -- the benchmark number looks excellent -- and it surfaces only under
the transformation suite, because the first JPEG re-encode overwrites the
quantisation tables and the first resize overwrites the dimensions. A detector
leaning on this cue does not degrade gracefully under transformation; it goes
to chance, because its entire signal was in the container.

This is also why `bias_match: true` is the training default in
`configs/default.yaml`, and why the crop pipeline works at native resolution:
the confound has to be destroyed before the model sees the data, not corrected
for afterwards.

## 3. Match the bias, and ask again

`bias_match` re-encodes every image through one encoder -- fixed quality 90,
fixed 4:2:0 subsampling -- and crops (never resizes) to a common 256x256.
Cropping matters: resampling would rewrite the high-frequency structure that a
real detector reads. Images shorter than 256 on an axis are reflect-padded and
counted.

Afterwards, every feature the probe uses is constant by construction except
bytes-per-pixel. If the probe still scores above chance, whatever is left is
what a shared encoder cannot erase.

In [4]:
result = compare(cfg, rows=rows)
print(format_table(result))
print()
print("match settings:", result["match"])

set                 n   ROC AUC  +/- fold
-----------------------------------------
raw              4000    1.0000    0.0000
bias-matched     4000    0.6527    0.0162
-----------------------------------------
removed                  0.3473

match settings: {'quality': 90, 'size': 256, 'subsampling': '4:2:0', 'padded_images': 20}


In [5]:
matched = result["rows"][-1]
print("strongest coefficients after matching:")
for f in matched["top_features"][:6]:
    print(f"    {f['name']:<22}{f['coef']:>8.3f}")

strongest coefficients after matching:
    bytes_per_pixel         -0.560
    has_chroma_table         0.000
    q_luma_42                0.000
    q_luma_48                0.000
    q_luma_47                0.000
    q_luma_46                0.000


### Interpretation

Matching removes about **0.35 AUC**. Everything the probe was leaning on --
subsampling, quantisation tables, resolution, container format -- collapses to
a coefficient of exactly zero, because those fields are now identical across
both classes. The confound was real, and it is gone.

What survives is a single feature: **bytes-per-pixel**, worth roughly 0.65 AUC
on its own. That is not a leak in the same sense. Once quality, subsampling and
dimensions are pinned, compressed size measures how compressible the *content*
is, and DALL-E images are smoother and less texturally dense than photographs.
It is a weak content statistic reached through the file size rather than the
pixels -- a genuine, if crude, signal about the images, not about who saved
them.

So the honest reading of the two rows is: **1.000 is the benchmark's
bookkeeping, 0.65 is roughly the floor a trivial content statistic reaches, and
anything a real detector claims above that is what it actually earned.** The
robustness table in `reports/` should be read against 0.65, not against 0.5.